# TVB Whole-Brain Workflow: V1/V2 Stimulus-Evoked Response

This notebook sets up a whole-brain simulation using TVB's default 76-region connectivity to study the propagation of a visual stimulus delivered to primary (V1) and secondary (V2) visual cortices.

**Steps implemented in this notebook:**
1. Import libraries and load default connectivity.
2. Configure a `Generic2dOscillator` model in a stable spiral regime (~10 Hz).
3. Define a region-wise stimulus (PulseTrain) targeting V1 and V2.
4. Assemble a stochastic Heun integrator and temporal-average monitor.
5. Build and configure the simulator.
6. Run the simulation for ~10 s.
7. Visualize time series with stimulus-onset annotation.
8. Verify the ~10 Hz regime claim with a power-spectrum estimate.
9. Compute post-stimulus functional connectivity.

## Step 1 – Imports and Connectivity

We use TVB’s default 76-region cortico-cortical connectivity (Desikan–Killiany parcellation). Region 35 corresponds to **rV1** and region 36 to **rV2**.

We also import `scipy.signal` for post-hoc Welch power-spectrum estimation.

In [ ]:
from tvb.simulator.lab import *
import numpy
import matplotlib.pyplot as plt
from scipy import signal

# Load default 76-region connectivity
conn = connectivity.Connectivity.from_file()
conn.configure()

print('Number of regions:', conn.number_of_regions)
print('V1 label:', conn.region_labels[35])
print('V2 label:', conn.region_labels[36])

## Step 2 – Model: Generic2dOscillator in Stable Spiral Regime (~10 Hz)

The `Generic2dOscillator` is a two-state-variable population model (`V`, `W`).

Rationale for the chosen parameter set:
- `a = -2.0` shifts the W-nullcline to create a stable focus.
- `b = -10.0` sets a steep linear restoring force that dampens perturbations.
- `c = 0.0` removes quadratic nonlinearity, keeping the dynamics near the linear regime.
- `d = 0.02` scales the intrinsic time constant so the small-perturbation period is ~100 ms (≈10 Hz).
- `I = 0.0` provides no external DC drive, letting the network rest at the fixed point.

The resulting fixed point is a **stable spiral** whose damped oscillation frequency is approximately **10 Hz**.
The remaining parameters (`tau`, `alpha`, `beta`, `gamma`, `e`, `f`, `g`) are kept at TVB defaults.

In [ ]:
model = models.Generic2dOscillator(
    a=numpy.array([-2.0]),   # vertical shift of W-nullcline
    b=numpy.array([-10.0]),  # linear slope of W-nullcline
    c=numpy.array([0.0]),    # quadratic term of W-nullcline
    d=numpy.array([0.02]),   # temporal scale factor (~10 Hz)
    I=numpy.array([0.0]),    # no external DC drive
)

print('Model state variables:', model.state_variables)
print('Number of state variables:', model._nvar)

## Step 3 – Stimulus: PulseTrain to V1 and V2

A brief external pulse is delivered to regions **35 (rV1)** and **36 (rV2)**.

Rationale:
- `onset = 500 ms` – pulse begins at 500 ms, providing a short pre-stimulus baseline.
- `tau = 5 ms` – short pulse width mimics a brief sensory event.
- `T = 10000 ms` – repetition period far longer than the simulation, so only a single pulse occurs.
- `amp = 1e-3` – low amplitude to perturb the stable spiral without pushing it into a saturated regime.

In [ ]:
# Stimulus weights: shape (n_regions, 1)
stim_weights = numpy.zeros((conn.number_of_regions, 1))
stim_weights[35] = 1.0   # V1
stim_weights[36] = 1.0   # V2

# Temporal profile: single brief pulse
eqn_t = equations.PulseTrain()
eqn_t.parameters['onset'] = 500.0   # ms
eqn_t.parameters['tau']   = 5.0     # ms
eqn_t.parameters['T']     = 10000.0 # ms (effectively a single pulse)
eqn_t.parameters['amp']   = 1e-3    # amplitude

stimulus = patterns.StimuliRegion(
    temporal=eqn_t,
    connectivity=conn,
    weight=stim_weights
)
stimulus.configure()

## Step 4 – Integrator and Noise

We use a **HeunStochastic** integrator with a small time step and weak additive noise.

Rationale:
- `dt = 2⁻⁶ ≈ 0.016 ms` is far smaller than the ~100 ms oscillation period and satisfies the stability requirement for the Heun scheme with additive noise.
- `nsig = [0.015, 0.015]` provides weak background fluctuations that keep the network near the fixed point without dominating the dynamics. One entry is required per state variable (`V`, `W`).

In [ ]:
hiss = noise.Additive(nsig=numpy.array([0.015, 0.015]))
heunint = integrators.HeunStochastic(dt=2**-6, noise=hiss)

print('Integrator dt:', heunint.dt, 'ms')

## Step 5 – Monitor, Coupling, and Simulator Assembly

Rationale:
- `TemporalAverage(period=5.0)` samples at 200 Hz, well above the Nyquist limit for the ~10 Hz oscillation and sufficient for PSD and FC estimation.
- `Linear(a=0.0154)` uses the default TVB coupling scaling for `Generic2dOscillator`; it yields weak inter-regional coupling so that propagation is driven primarily by structural connectivity rather than global synchronization.
- The stimulus is passed directly to the `Simulator` constructor.

In [ ]:
mon = (monitors.TemporalAverage(period=5.0),)

coup = coupling.Linear(a=numpy.array([0.0154]))

sim = simulator.Simulator(
    model=model,
    connectivity=conn,
    coupling=coup,
    integrator=heunint,
    monitors=mon,
    stimulus=stimulus
)
sim.configure()

print('Simulator configured successfully.')
print('Simulation length will be set at runtime.')

## Step 6 – Run Simulation (~10 s)

A run length of **10 000 ms** provides 500 ms of pre-stimulus baseline and 9.5 s of post-stimulus dynamics, enough to estimate power spectra and observe stimulus-evoked propagation.

Because the stimulus arrives at 500 ms, we do **not** apply a long burn-in; the brief initial transient (<100 ms) is retained as part of the pre-stimulus baseline.

In [ ]:
(t, y), = sim.run(simulation_length=10000.0)
print('Time points:', t.shape)
print('Data shape:', y.shape)  # (time, state-vars, regions, modes)

## Step 7 – Time-Series Plots with Stimulus-Onset Annotation

We plot the `V` state variable for V1, V2, and a few representative downstream regions. A vertical dashed line marks the stimulus onset at **500 ms**.

In [ ]:
# Extract V variable for all regions: shape (time, regions)
ts_v = y[:, 0, :, 0].squeeze()

# Select a few regions to plot
regions_to_plot = [35, 36, 0, 10]  # V1, V2, and two other cortical regions
labels = [conn.region_labels[r] for r in regions_to_plot]

plt.figure(figsize=(12, 5))
for idx, r in enumerate(regions_to_plot):
    plt.plot(t, ts_v[:, r], label=labels[idx], alpha=0.8)

plt.axvline(x=500.0, color='red', linestyle='--', linewidth=1.5, label='stimulus onset')
plt.xlabel('Time (ms)')
plt.ylabel('V (a.u.)')
plt.title('Region-wise time series (V variable)')
plt.legend(loc='upper right')
plt.xlim(t.min(), t.max())
plt.tight_layout()
plt.show()

## Step 8 – Regime Verification: Power Spectrum of V1

Before calling the dynamics a "~10 Hz stable spiral" we verify the claim empirically. We compute a Welch PSD on a **post-stimulus window** (1000–10 000 ms) that avoids the immediate stimulus transient while still capturing the ongoing intrinsic dynamics. The sampling frequency is 200 Hz (period = 5 ms).

In [ ]:
# Post-stimulus analysis window (avoid the immediate 500 ms transient)
psd_mask = (t >= 1000.0) & (t < 10000.0)
v1_post = ts_v[psd_mask, 35]  # V1 channel

fs = 1000.0 / mon[0].period  # 200 Hz
freqs, psd = signal.welch(v1_post, fs=fs, nperseg=256)
peak_idx = numpy.argmax(psd)
peak_freq = freqs[peak_idx]

plt.figure(figsize=(8, 4))
plt.semilogy(freqs, psd, label='V1 PSD')
plt.axvline(x=peak_freq, color='orange', linestyle='--', label=f'peak = {peak_freq:.2f} Hz')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.title('Welch PSD of V1 (1000–10 000 ms post-stimulus)')
plt.legend()
plt.xlim(0, 50)
plt.tight_layout()
plt.show()

print(f'Empirical peak frequency: {peak_freq:.2f} Hz')

## Step 9 – Post-Stimulus Functional Connectivity

To assess stimulus-evoked network interactions, we compute Pearson correlation across all 76 regions in the window **500–1500 ms** (the first second after stimulus onset). This window aligns the analysis with the scientific question of propagation rather than mixing in long pre-stimulus baseline.

In [ ]:
# Extract the evoked window
evoked_mask = (t >= 500.0) & (t < 1500.0)
ts_evoked = ts_v[evoked_mask, :]  # shape (time, regions)

fc = numpy.corrcoef(ts_evoked.T)

plt.figure(figsize=(7, 6))
plt.imshow(fc, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Pearson r')
plt.title('Functional Connectivity (500–1500 ms post-stimulus)')
plt.xlabel('Region')
plt.ylabel('Region')
plt.tight_layout()
plt.show()

# Exclude diagonal for mean FC
mask_offdiag = ~numpy.eye(fc.shape[0], dtype=bool)
mean_fc = fc[mask_offdiag].mean()
print(f'Mean off-diagonal FC: {mean_fc:.4f}')